# First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

notebook นี้เปิดดูข้อมูลครั้งแรก **หลังจาก** commit คำถามส่วนที่ 1 ใน `questions.md` แล้ว (commit `180e066`) เป้าหมายคือทำความเข้าใจรูปร่างของข้อมูล ยังไม่ตอบคำถามเชิงวิเคราะห์

## ส่วนที่ 1: ข้อมูลปัจจุบันจาก Air4Thai

In [1]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # แก้ SSLError จาก certificate chain ของ Air4Thai

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]
print(len(stations))
stations[0]

<class 'dict'> ['stations']
180


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-24',
  'time': '13:00',
  'PM25': {'color_id': '1', 'aqi': '11', 'value': '6.7'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '19', 'value': '27'},
  'CO': {'color_id': '1', 'aqi': '4', 'value': '0.70'},
  'NO2': {'color_id': '1', 'aqi': '0', 'value': '1'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '19', 'param': 'O3'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()

(180, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

In [4]:
# ชนิดข้อมูลของคอลัมน์ และตัวอย่าง 3 แถวแรก
print(df_now.dtypes.value_counts())
df_now.head(3).T

object    32
Name: count, dtype: int64


,0,1,2
stationID,119t,02t,03t
nameTH,สวนสาธารณะธารา,มหาวิทยาลัยราชภัฏบ้านสมเด็จเจ้าพระยา,ริมถนนทางหลวงหมายเลข 3902
nameEN,Thara Public Park,Bansomdejchaopraya Rajabhat University,Highway NO.3902 km.13 +600
areaTH,"ต.ปากน้ำ อ.เมือง, กระบี่","แขวงหิรัญรูจี เขตธนบุรี, กรุงเทพฯ","ริมถนนกาญจนาภิเษก เขตบางขุนเทียน, กรุงเทพฯ"
areaEN,"Pak Nam Subdistrict, Mueang District, Krabi","Hiran Ruchi, Khet Thon Buri, Bangkok","Kanchanaphisek Rd, Bang Khun Thian, Bangkok"
stationType,GROUND,GROUND,GROUND
lat,8.0506237,13.732846,13.636514
long,98.9180489,100.487662,100.414262
forecast,[],[],[]
AQILast.date,2026-11-24,2026-11-24,2026-11-24


### สิ่งที่เห็นจากข้อมูลปัจจุบัน (ดึงเมื่อ 24 พ.ย. 2569 ตัวเลขจะเปลี่ยนทุกครั้งที่รันใหม่)

1. **หนึ่งแถวของ `df_now` คืออะไร:** หนึ่งสถานี ณ เวลาวัดล่าสุดของสถานีนั้น ได้ราว 180 สถานี 32 คอลัมน์ แต่ "เวลาล่าสุด" ของแต่ละสถานีไม่จำเป็นต้องตรงกัน บางสถานีค่าล่าสุดเก่าหลายสัปดาห์ (ดู cell ถัดไป)
2. **คอลัมน์สำคัญ:** รหัสสถานี `stationID` · ชื่อ `nameTH` / `nameEN` · พื้นที่ `areaTH` / `areaEN` (จังหวัดอยู่ท้ายข้อความ ไม่ได้แยกคอลัมน์) · พิกัด `lat`, `long` · วันที่ `AQILast.date` · เวลา `AQILast.time` (แยกกันคนละคอลัมน์) · ประเภทสถานี `stationType`
3. **ค่า PM2.5:** มีทั้งค่าความเข้มข้น `AQILast.PM25.value` (หน่วยน่าจะเป็น µg/m³ แต่ใน JSON ไม่ได้ระบุหน่วยไว้) และค่าดัชนี `AQILast.PM25.aqi` กับระดับสี `AQILast.PM25.color_id` ส่วน `AQILast.AQI` เป็นดัชนีรวมของสถานี โดย `AQILast.AQI.param` บอกว่ามลพิษตัวไหนเป็นตัวกำหนด
4. **ชนิดข้อมูลและค่าแปลก:** ทุกคอลัมน์เป็น `object` (ตัวเลขถูกส่งมาเป็นข้อความ) ต้องแปลงด้วย `pd.to_numeric` และพบค่า `-1` ซึ่งน่าจะใช้แทนค่าหาย คอลัมน์ `forecast` เป็น list ว่าง

In [5]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 1


count    180.000000
mean      23.053889
std        8.770235
min       -1.000000
25%       17.000000
50%       23.100000
75%       29.750000
max       47.200000
Name: AQILast.PM25.value, dtype: float64

In [6]:
# ค่า -1 ปรากฏในคอลัมน์ค่าความเข้มข้นอื่นด้วยหรือไม่ (น่าจะใช้แทนค่าหาย)
value_cols = [c for c in df_now.columns if c.endswith(".value")]
print((df_now[value_cols].apply(pd.to_numeric, errors="coerce") == -1).sum())

# เวลาล่าสุดของแต่ละสถานีตรงกันหรือไม่ และประเภทสถานี
print(df_now.groupby(["AQILast.date", "AQILast.time"]).size())
print(df_now["stationType"].value_counts())

AQILast.PM25.value      1
AQILast.PM10.value    114
AQILast.O3.value      152
AQILast.CO.value      150
AQILast.NO2.value     147
AQILast.SO2.value     148
dtype: int64
AQILast.date  AQILast.time
2026-10-09    15:00            1
2026-10-21    09:00            1
2026-11-11    14:00            1
2026-11-18    10:00            1
2026-11-21    18:00            1
2026-11-24    11:00            1
              12:00            3
              13:00           94
              14:00           77
dtype: int64
stationType
GROUND    104
BKK        68
MOBILE      8
Name: count, dtype: int64


**ค่าหายและค่าแปลก:**

- PM2.5 แปลงเป็นตัวเลขได้ทุกแถว แต่มีค่า `-1` อยู่ 1 สถานี ซึ่งเป็นไปไม่ได้สำหรับความเข้มข้น จึงน่าจะเป็นรหัสแทนค่าหาย (ตรงกับ README ของไฟล์ตัวอย่างที่บอกว่าค่าติดลบใน API แทนข้อมูลหาย) ถ้าไม่แปลงเป็นค่าว่างก่อน ค่าเฉลี่ยจะต่ำกว่าจริง
- มลพิษอื่นมี `-1` มาก (PM10 ราว 114 จาก 180 สถานี, O3/CO/NO2/SO2 ราว 150) แปลว่าหลายสถานีไม่ได้วัดมลพิษเหล่านี้ PM2.5 เป็นตัวแปรที่ครบที่สุด
- เวลาล่าสุดของสถานีส่วนใหญ่อยู่ในชั่วโมงเดียวกัน แต่มีบางสถานีที่ค่าล่าสุดเก่าหลายวันถึงหลายสัปดาห์ ถ้านำค่าปัจจุบันไปแสดงบนแผนที่ ควรตรวจอายุของข้อมูลก่อน
- ประเภทสถานี: `GROUND` (สถานีภาคพื้นดิน), `BKK` (สถานีของกรุงเทพฯ) และ `MOBILE` (สถานีเคลื่อนที่) ควรตรวจสอบต่อว่าสถานีเคลื่อนที่ควรรวมในการวิเคราะห์หรือไม่

> ข้อมูลปัจจุบันมีแค่เวลาเดียวต่อสถานี จึงดูช่วงเวลาและความถี่ไม่ได้ ใช้ไฟล์ตัวอย่างย้อนหลังในส่วนที่ 2 แทน

## ส่วนที่ 2: ไฟล์ตัวอย่างย้อนหลัง (เฉพาะช่วงสำรวจ 2563–2566)

ใช้เฉพาะข้อมูลก่อน 2024-01-01 ห้ามดูช่วงทดสอบ 2567–2568

In [7]:
hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ
explore = hourly[hourly["ts"] < "2024-01-01"]

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(0, 8) 0 สถานี
NaT → NaT
Series([], Name: count, dtype: int64)


station_id   NaN
ts           NaN
pm25         NaN
pm10         NaN
o3           NaN
co           NaN
no2          NaN
so2          NaN
dtype: float64

**ข้อสังเกตสำคัญ:** ตัวกรองช่วงสำรวจ (`ts < 2024-01-01`) ได้ **0 แถว** เพราะไฟล์ตัวอย่างครอบคลุมแค่ 1 ก.ย. – 17 พ.ย. 2569 (README ของ `sample-data/` บอกว่า API ย้อนหลังของ Air4Thai เก็บข้อมูลประมาณ 3 เดือน) ไฟล์นี้ไม่มีทั้งช่วงสำรวจ 2563–2566 และช่วงทดสอบ 2567–2568 จึงใช้ทั้งไฟล์เป็นข้อมูลสำรวจใน first look นี้ โดยมี `assert` ยืนยันว่าไม่มีข้อมูลช่วงทดสอบปนอยู่

In [8]:
# ช่วงเวลาจริงของไฟล์ทั้งไฟล์
print(hourly["ts"].min(), "→", hourly["ts"].max())

# ไฟล์อยู่นอกทั้งช่วงสำรวจและช่วงทดสอบ (ไม่มีข้อมูลปี 2567–2568) จึงใช้ทั้งไฟล์เป็นข้อมูลสำรวจ
assert not hourly["ts"].between("2024-01-01", "2025-12-31 23:59").any(), "มีข้อมูลช่วงทดสอบปนอยู่"
explore = hourly.copy()
print(explore.shape, explore["station_id"].nunique(), "สถานี")
explore.isna().mean().round(3)

2026-09-01 00:00:00 → 2026-11-17 23:00:00
(333216, 8) 178 สถานี


station_id    0.000
ts            0.000
pm25          0.071
pm10          0.646
o3            0.849
co            0.811
no2           0.794
so2           0.828
dtype: float64

In [9]:
# ความถี่ที่ถูกต้องต้องดูภายในสถานีเดียวกัน (diff ข้ามสถานีจะได้ค่าติดลบ)
explore.sort_values(["station_id", "ts"]).groupby("station_id")["ts"].diff().value_counts().head(3)

ts
0 days 01:00:00    333038
Name: count, dtype: int64

In [10]:
# โครงสร้างไฟล์ทั้งสอง
print(explore.dtypes, end='\n\n')
print(st.shape)
st.head(3)

station_id            object
ts            datetime64[ns]
pm25                 float64
pm10                 float64
o3                   float64
co                   float64
no2                  float64
so2                  float64
dtype: object

(178, 8)


,station_id,name_th,name_en,province,region,lat,lon,station_type
0,119t,สวนสาธารณะธารา,Thara Public Park,กระบี่,ใต้,8.050624,98.918049,GROUND
1,02t,มหาวิทยาลัยราชภัฏบ้านสมเด็จเจ้าพระยา,Bansomdejchaopraya Rajabhat University,กรุงเทพฯ,กลาง,13.732846,100.487662,GROUND
2,03t,ริมถนนทางหลวงหมายเลข 3902,Highway NO.3902 km.13 +600,กรุงเทพฯ,กลาง,13.636514,100.414262,GROUND


In [11]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

station_id
o67    0.986645
o29    0.942842
o59    0.933226
o64    0.923077
o68    0.905449
76t    0.803419
74t    0.748932
18t    0.705128
75t    0.704060
26t    0.618056
Name: pm25, dtype: float64

In [12]:
# สัดส่วนค่าหายของ pm25 แยกตามเดือน (ดูว่าค่าหายกระจุกในบางเดือนหรือไม่)
explore.groupby(explore["ts"].dt.month)["pm25"].apply(lambda s: s.isna().mean()).round(3)

ts
9     0.060
10    0.086
11    0.064
Name: pm25, dtype: float64

In [13]:
# สถานีในพื้นที่ที่คำถามส่วนที่ 1 ต้องใช้ (Q1: เชียงใหม่, Q2–Q3: กรุงเทพฯ)
st[st["province"].isin(["กรุงเทพฯ", "เชียงใหม่"])].groupby("province").size()

province
กรุงเทพฯ     79
เชียงใหม่     3
dtype: int64

In [14]:
# ทุกสถานีในไฟล์รายชั่วโมงมีอยู่ในตารางสถานีหรือไม่
print("สถานีที่ไม่มีในตาราง stations:", set(explore["station_id"]) - set(st["station_id"]))
print("จำนวนแถวต่อสถานี:", explore.groupby("station_id").size().describe()[["min", "max"]].tolist())

สถานีที่ไม่มีในตาราง stations: set()
จำนวนแถวต่อสถานี: [1872.0, 1872.0]


In [15]:
# ค่าผิดปกติ: ค่าติดลบ และค่าสูงมาก
print("ค่าติดลบ:", (explore["pm25"] < 0).sum())
explore["pm25"].describe(percentiles=[.5, .9, .99])

ค่าติดลบ: 0


count    309435.000000
mean         15.241451
std           8.708748
min           0.000000
50%          13.600000
90%          26.400000
99%          43.500000
max         167.900000
Name: pm25, dtype: float64

### สรุปสิ่งที่เห็น

1. **พบว่า** ไฟล์ตัวอย่างมีแค่ ก.ย.–พ.ย. 2569 ไม่มีเดือน ธ.ค.–เม.ย. เลย **ควรตรวจสอบต่อ** ว่าจะหาข้อมูลย้อนหลังปี 2563–2566 จากแหล่งใด เพราะ Q1 (ก.พ.–เม.ย.), Q2 (ธ.ค.–ก.พ.) และ Q3 (ทุกเดือน) ตอบด้วยไฟล์นี้ไม่ได้
2. **พบว่า** ข้อมูลเป็นรายชั่วโมงสม่ำเสมอ ทุกสถานีมี 1,872 แถวเท่ากัน (78 วัน × 24 ชั่วโมง) ช่วงห่างภายในสถานีเป็น 1 ชั่วโมงทั้งหมด ค่าที่หายไปจึงอยู่ในรูปค่าว่างของ `pm25` ไม่ใช่แถวที่หายไป ทั้ง 178 สถานีมีอยู่ในตาราง `stations`
3. **พบว่า** `pm25` หายราว 7% ของทั้งไฟล์ แต่ค่าหายกระจุกในบางสถานี (บางสถานีหายเกิน 90% และหลายสถานีที่หายมากมีรหัสขึ้นต้นด้วย `o`) **ควรตรวจสอบต่อ** ว่าสถานีกลุ่มนี้เป็นสถานีประเภทใด และควรตัดออกหรือไม่ ส่วนมลพิษอื่นหายมาก (PM10 ราว 65%, O3/CO/NO2/SO2 ราว 80%)
4. **พบว่า** `pm25` ไม่มีค่าติดลบในไฟล์นี้ (ถูกแปลงเป็นค่าว่างแล้ว) ค่ามัธยฐานราว 14 µg/m³ แต่ค่าสูงสุดราว 168 µg/m³ ซึ่งสูงกว่าเปอร์เซ็นไทล์ที่ 99 (ราว 44) มาก **ควรตรวจสอบต่อ** ว่าเป็นเหตุการณ์จริงหรือความผิดพลาดของเครื่องวัด
5. **พบว่า** สถานีกระจุกในกรุงเทพฯ (79 จาก 178 สถานี) ขณะที่เชียงใหม่มีเพียง 3 สถานี **ควรตรวจสอบต่อ** ว่า 3 สถานีมีข้อมูลครบพอสำหรับ Q1 หรือไม่ และการเฉลี่ยทั้งประเทศจะถูกกรุงเทพฯ ถ่วงน้ำหนักมากเกินไปหรือไม่